# Quantum Phase Estimation of a Qubitized Hamiltonian Workbook

This workbook describes the solutions to the problems offered in the [Quantum Phase Estimation of a Qubitized Hamiltonian kata](./QubitizedHamiltonianQPE_1_Hamiltonian.ipynb) (all parts). Since the tasks are offered as programming problems, the explanations also cover some elements of Workbench that might be non-obvious for a first-time user.

In [1]:
from psiqdk.workbench import Qubits, Qubrick, units
from math import atan2, sqrt

## Part II. Block Encoding

Solutions to problems and exercises from [Part II](./QubitizedHamiltonianQPE_2_BlockEncoding.ipynb).

### Exercise 1. State preparation in Workbench

This program pretty much follows the steps outlined in [How to Write a Simple Program](https://construct.psiquantum.com/docs/psiqdk-workbench/howtos/Write-Simple-Program.html).

```python
from psiqdk.workbench import QPU, Qubits
from psiqdk.algorithms import ArbitraryStatePrep

# Create a QPU object with 2 qubits
qpu = QPU(num_qubits=2)

# Allocate a fresh register with 2 qubits
reg = Qubits(2, 'reg', qpu)

# Instantiate the state preparation Qubrick
state_prep = ArbitraryStatePrep([0.6, 0.48, 0.64])

# Prepare the state using plain (uncontrolled) variant of the unitary
state_prep.compute(reg)

# Print the state vector
qpu.print_state_vector()

# Draw the circuit
qpu.draw(show_qubricks=True)
```

### Problem 1. Load data

The most naive implementation of data loading uses multiple multi-controlled multi-target X gates, one per item in `data`.

In Workbench, [multi-target gates](https://construct.psiquantum.com/docs/psiqdk-workbench/new-tutorials/Basic-Gates.html#applying-a-gate-to-a-subset-of-qubits-using-bit-masks) are single-qubit gates (in our case, the X gate) which are applied to each qubit in the given subset of a register. To apply a multi-target gate, you can pass a bit mask as the `tgt_bits` argument; the gate will be applied to only qubits in the positions where the bit string has $1$ bits. In other words, applying a multi-target X gate with `tgt_bits=data[j]` to a register in the $\ket{0}$ state writes the value `data[j]` into this register.

How can we use this for data loading? For an element of `data` with index $j$, we use a controlled gate with the control pattern being the index $j$ itself and the target bit mask encoding the value of that element `data[j]`. Here both the index and the value are expressed in binary using little-endian encoding.

To implement the controlled variant of data loading, we need to append the control register `ctrl` to the register used as the control for the gate.

In [3]:
from psiqdk.workbench import Qubits, Qubrick

class DataLoader(Qubrick):
    def _compute(self, index: Qubits, target: Qubits, data: list[int], ctrl: Qubits | None = None) -> None:
        """Load the data into target register, indexed by the value of the index register."""
        for j in range(len(data)):
            target.x(data[j], cond=(index == j) | ctrl)

### Problem 2. Add two unsigned integers

The main building block of the naive addition algorithm is increment - an operation that increments a register of length $N$ modulo $2^N$. The simplest implementation of increment works as follows.

We start with the most significant bit (in the case of little-endian notation used by Workbench, this is the last bit of the register) and work our way down to the least significant bit (the first bit of the register). On each step, the current bit is incremented only if all less significant bits are $1$ - in this case, when we add $1$ to the original number, all of those bits are incremented with a carry bit $1$, and the overflow will carry into the current bit. To do this, we can use a controlled X gate, with the current qubit as the target and all less significant qubits as controls. The last bit to increment will be the least significant bit, which is incremented using a plain X gate.

To implement the controlled variant of increment with control register `ctrl`, we simply need to append the control register to each gate.

Now we can implement the addition algorithm itself. Here is the outline of the solution:

* To add a constant to a number, we can represent that constant as a sum of powers of $2$ and add each of those powers to the number separately. 
* Adding $2^k$ to a number can be represented as incrementing this number without its $k-1$ least significant digits, that is, incrementing a sub-register of the register that stores this number.
* To add a quantum number to another quantum number, we can perform a sequence of controlled increments, with each of the digits of the addend acting as control to specify whether the corresponding increment needs to be performed.

> For a detailed step-by-step walkthrough of the solution, see part 1 of the [Arithmetic Data Types](https://github.com/PsiQ/workbench-quantum-katas/blob/main/ArithmeticDataTypes/ArithmeticDataTypes.ipynb) kata.

In [ ]:
from psiqdk.workbench import Qubits, Qubrick

class Adder(Qubrick):
    def _increment(self, a: Qubits, ctrl: Qubits | None = None) -> None:
        """Increment a register, with an optional quantum control."""
        for ind in range(len(a) - 1, 0, -1):
            a[ind].x(cond=a[:ind] | ctrl)
        a[0].x(cond=ctrl)

    def _compute(self, a: Qubits, b: Qubits, ctrl: Qubits | None = None, **kwargs) -> None:
        """Add register b to register a in-place, with an optional quantum control."""
        m = len(b)
        for ind in range(m):
            self._increment(a[ind:], b[ind] | ctrl)

### QRE exercises (Part 4)

Reference solutions for the resource-estimation exercises in [Part 4](QubitizedHamiltonianQPE_4_QRE.ipynb): the two adder swaps (`exercise_1`, `exercise_2`) and the mocked `AbstractRoutine`. The `build_qpu` helper is rebuilt here so the solutions are self-contained.

In [6]:
import numpy as np
from psiqdk.workbench import QPU, Qubits, Qubrick
from psiqdk.workbench.qft import QFT
from psiqdk.algorithms import PrepareNaive, SelectNaive, NaiveAdd, GidneyAdd, CuccaroAdd, QPE
from psiqdk.workbench.experimental.symbolics import QubrickCosts


class HamiltonianEncoding(Qubrick):
    """Qubitization walk operator (block encoding + reflection) for a hopping
    Hamiltonian specified by aligned (coefficient, offset) term pairs.

    The offsets  and the addends-register width are passed in, so the
    same class works for any number of terms / any lattice size.
    """

    def __init__(self, prepare, qrom, adder, offsets, addends_size, **kwargs):
        self.prepare = prepare
        self.qrom = qrom
        self.adder = adder
        self.offsets = offsets
        self.addends_size = addends_size
        super().__init__(**kwargs)

    def _compute(self, state_reg, coefficient_reg, add_subtract_reg, ctrl=0):
        # PREPARE
        self.prepare.compute(coefficient_reg, cond=ctrl)
        # QROM (uncomputed automatically on context exit)
        addends = self.alloc_temp_qreg(self.addends_size, "addends")
        with self.qrom.computed(coefficient_reg, addends, self.offsets, ctrl=ctrl):
            add_subtract_reg.had(cond=ctrl)
            add_subtract_reg.x(cond=ctrl)  # makes SELECT self-inverse
            # add on the |0> branch, subtract (adder dagger) on the |1> branch
            self.adder.compute(state_reg, addends, ctrl=(add_subtract_reg == 0) | ctrl)
            self.adder.compute(state_reg, addends, ctrl=(add_subtract_reg == 1) | ctrl, dagger=True)
            add_subtract_reg.had(cond=ctrl)
        addends.release()
        # PREPARE dagger
        self.prepare.uncompute()
        # reflection for qubitization
        (~coefficient_reg | ~add_subtract_reg).reflect(ctrl=ctrl)
        if ctrl:
            ctrl.reflect()


def build_qpu(
    state_size,
    coefficients,
    offsets,
    bits_of_precision=5,
    prepare=None,
    qrom=None,
    adder=None,
    buffer=10,
):
    """Build a QPU holding a full QPE with the qubitized Hamiltonian.

    Args:
        state_size: qubits in the system register (lattice has 2**state_size sites).
        coefficients: term weights w_j; the default PREPARE loads sqrt of these.
        offsets: integer offsets loaded by the QROM and added to / subtracted from the
            state register. One per coefficient.
        bits_of_precision: size of the QPE phase register.
        prepare: state-preparation Qubrick. Defaults to PrepareNaive.
        qrom: data-lookup Qubrick. Defaults to SelectNaive.
        adder: adder Qubrick. Defaults to NaiveAdd.
        buffer: spare qubits for ancilla-heavy adder decompositions.

    """
    assert len(coefficients) == len(offsets), "need one coefficient per offsets term"
    assert all(int(d) == d and d >= 0 for d in offsets), "offsets must be non-negative integers"
    assert max(offsets) < 2 ** state_size, "an offset does not fit in the state register"

    if prepare is None:
        prepare = PrepareNaive([np.sqrt(w) for w in coefficients])
    if qrom is None:
        qrom = SelectNaive()
    if adder is None:
        adder = NaiveAdd()

    coefficient_size = int(np.ceil(np.log2((len(coefficients)))))   # QROM address bits
    addends_size = int(np.ceil(np.log2(max(offsets)))) + 1           # bits to hold the largest offset
    ctrl_size = 1

    num_qubits = (
        state_size + coefficient_size + ctrl_size + bits_of_precision + addends_size + buffer
    )

    qpu = QPU(num_qubits=num_qubits, filters=[">>buffer>>"])
    state_reg = Qubits(state_size, "psi", qpu)
    coefficient_reg = Qubits(coefficient_size, "coefficient_reg", qpu)
    add_subtract_reg = Qubits(1, "add_subtract_reg", qpu)
    phases_reg = Qubits(bits_of_precision, "phases_reg", qpu)

    qft = QFT()
    qft.compute(state_reg)

    encoding = HamiltonianEncoding(
        prepare=prepare,
        qrom=qrom,
        adder=adder,
        offsets=offsets,
        addends_size=addends_size,
    )

    qpe = QPE(unitary=encoding, bits_of_precision=bits_of_precision)
    qpe.compute(
        state_reg,
        phases_reg,
        coefficient_reg=coefficient_reg,
        add_subtract_reg=add_subtract_reg,
    )
    return qpu

In [7]:
# Config matching Part 4's exercise cells.
state_size = 8
ws = [0.1] * 8
offsets = list(range(1, 9))
bits_of_precision = 5
BUFFER = 20  # headroom so every candidate adder can allocate its ancillas


def exercise_1():
    # NaiveAdd sits around 500k active volume. GidneyAdd is the cheapest of the
    # drop-in adders and brings the active volume below 260,000.
    return build_qpu(
        state_size, coefficients=ws, offsets=offsets,
        bits_of_precision=bits_of_precision, buffer=BUFFER, adder=GidneyAdd(),
    )


def exercise_2():
    # Tighter brief: active volume < 280,000 AND qubit_highwater <= 26. GidneyAdd has the
    # lowest active volume but needs too many qubits; CuccaroAdd is the one that fits both.
    return build_qpu(
        state_size, coefficients=ws, offsets=offsets,
        bits_of_precision=bits_of_precision, buffer=BUFFER, adder=CuccaroAdd(),
    )


In [8]:
class AbstractRoutine(Qubrick):
    """active_volume ~ N**2 (declared cost); qubit_highwater ~ ceil(log2 N) (real ancillas)."""

    def __init__(self, N, **kwargs):
        self.N = N
        super().__init__(**kwargs)

    def _compute(self):
        # Real ancilla allocation -> drives qubit_highwater.
        self.alloc_temp_qreg(int(np.ceil(np.log2(self.N))), "temp_qubits")
        # Declared cost -> drives active_volume.
        self.get_qc().add_cost_event(QubrickCosts(active_volume=self.N ** 2))


> Copyright (c) 2026 PsiQuantum